In [1]:
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, "..")
from src.forecasting_utils import (
    all_metrics, TRAIN_END, VAL_END, TEST_END,
)

from statsforecast import StatsForecast
from statsforecast.models import (
    HistoricAverage,                # 28-day moving average proxy via window
    SimpleExponentialSmoothing,     # SES
    AutoARIMA,                      # ARIMA / SARIMA auto-selection
    CrostonClassic, CrostonSBA,     # intermittent demand
    SeasonalNaive,                  # honest baseline
)

warnings.filterwarnings("ignore")

In [2]:
# statsforecast expects columns: unique_id, ds, y
df = pd.read_parquet("../data/processed/subsample_features.parquet")
sf_df = df[["id", "date", "sales"]].rename(
    columns={"id": "unique_id", "date": "ds", "sales": "y"}
).sort_values(["unique_id", "ds"]).reset_index(drop=True)

# Train ends 2015-12-31; horizon = number of test days
train_df = sf_df[sf_df["ds"] <= TRAIN_END]
val_df   = sf_df[(sf_df["ds"] > TRAIN_END) & (sf_df["ds"] <= VAL_END)]
test_df  = sf_df[(sf_df["ds"] > VAL_END)   & (sf_df["ds"] <= TEST_END)]

horizon = test_df.groupby("unique_id").size().max()
print(f"Series: {sf_df['unique_id'].nunique()}  | Test horizon: {horizon} days")
print(f"Train rows: {len(train_df):,}  Test rows: {len(test_df):,}")

Series: 502  | Test horizon: 55 days
Train rows: 874,986  Test rows: 27,610


In [3]:
# statsforecast trains all models across all series in parallel
models = [
    HistoricAverage(),
    SimpleExponentialSmoothing(alpha=0.1),
    AutoARIMA(season_length=7),     # weekly seasonality → SARIMA when warranted
    CrostonClassic(),
    CrostonSBA(),
    SeasonalNaive(season_length=7),
]

sf = StatsForecast(models=models, freq="D", n_jobs=-1)

# Fit on train+val combined (no hyperparameter tuning needed for these)
fit_df = sf_df[sf_df["ds"] <= VAL_END]
print("Fitting baselines... (this can take 5-15 min depending on cores)")
forecasts = sf.forecast(df=fit_df, h=horizon)
forecasts = forecasts.reset_index()
print(forecasts.head())
print("Models produced columns:", [c for c in forecasts.columns if c not in ["unique_id", "ds"]])

Fitting baselines... (this can take 5-15 min depending on cores)


/Users/desmond/Capstone Project/retail-demand-forecasting/retail_forecasting_env/lib/python3.11/site-packages/scipy/optimize/_numdiff.py:619: RuntimeWarning: overflow encountered in divide
  J_transposed[i] = df / dx
/Users/desmond/Capstone Project/retail-demand-forecasting/retail_forecasting_env/lib/python3.11/site-packages/statsforecast/arima.py:1882: UserWarning: Stepwise search was stopped early due to reaching the model number limit: nmodels=94
  warnings.warn(
/Users/desmond/Capstone Project/retail-demand-forecasting/retail_forecasting_env/lib/python3.11/site-packages/statsforecast/arima.py:1882: UserWarning: Stepwise search was stopped early due to reaching the model number limit: nmodels=94
  warnings.warn(


   index                    unique_id         ds  HistoricAverage       SES  \
0      0  FOODS_1_002_TX_2_validation 2016-03-01         0.241819  0.069064   
1      1  FOODS_1_002_TX_2_validation 2016-03-02         0.241819  0.069064   
2      2  FOODS_1_002_TX_2_validation 2016-03-03         0.241819  0.069064   
3      3  FOODS_1_002_TX_2_validation 2016-03-04         0.241819  0.069064   
4      4  FOODS_1_002_TX_2_validation 2016-03-05         0.241819  0.069064   

   AutoARIMA  CrostonClassic  CrostonSBA  SeasonalNaive  
0  -0.021756        0.164783    0.156544            0.0  
1  -0.000277        0.164783    0.156544            0.0  
2  -0.002181        0.164783    0.156544            0.0  
3   0.009300        0.164783    0.156544            0.0  
4   0.060556        0.164783    0.156544            0.0  
Models produced columns: ['index', 'HistoricAverage', 'SES', 'AutoARIMA', 'CrostonClassic', 'CrostonSBA', 'SeasonalNaive']


In [4]:
# Merge forecasts with test actuals on (unique_id, ds)
merged = test_df.merge(forecasts, on=["unique_id", "ds"], how="inner")
print(f"Aligned test rows: {len(merged):,}")

model_cols = {
    "HistoricAverage":          "MA",
    "SimpleExponentialSmoothing": "SES",
    "AutoARIMA":                "ARIMA",
    "CrostonClassic":           "Croston",
    "CrostonSBA":               "Croston_SBA",
    "SeasonalNaive":            "SeasonalNaive",
}

# Some models can produce negative forecasts on sales — clip at 0
for col in model_cols:
    if col in merged.columns:
        merged[col] = merged[col].clip(lower=0)

results = []
for raw_col, label in model_cols.items():
    if raw_col in merged.columns:
        results.append(all_metrics(merged["y"], merged[raw_col], label))

results_df = pd.DataFrame(results)
results_df.to_csv("../data/processed/baseline_results_full.csv", index=False)
display(results_df)

Aligned test rows: 27,610


,model,MAE,RMSE,MAPE_pct,WMAPE_pct,Pred10_pct
0,MA,1.080141,2.554272,64.361831,83.457451,5.548409
1,ARIMA,0.997096,2.054741,58.447490,77.041001,8.532716
2,Croston,1.033128,2.082374,58.107241,79.825023,8.850013
3,Croston_SBA,1.020811,2.088825,57.634707,78.873321,7.417889
4,SeasonalNaive,1.144803,2.341553,81.867816,88.453574,16.079238


In [5]:
keep = ["unique_id", "ds", "y"] + [c for c in model_cols if c in merged.columns]
out = merged[keep].rename(columns={**model_cols, "unique_id": "id", "ds": "date", "y": "y_true"})
out.to_parquet("../artifacts/baseline_test_predictions.parquet", index=False)
print("Saved:", out.shape)

Saved: (27610, 8)


In [6]:
ml = pd.read_csv("../data/processed/ml_model_results.csv")
all_results = pd.concat([results_df, ml], ignore_index=True).sort_values("MAPE_pct")
display(all_results)

# Compute % improvement of best ML model over best classical baseline
best_classical_mape = results_df["MAPE_pct"].min()
best_ml_mape        = ml["MAPE_pct"].min()
improvement = (best_classical_mape - best_ml_mape) / best_classical_mape * 100
print(f"\nBest classical MAPE: {best_classical_mape:.2f}%")
print(f"Best ML MAPE:        {best_ml_mape:.2f}%")
print(f"MAPE improvement:    {improvement:+.1f}%  (target: 10-25%)")

,model,MAE,RMSE,MAPE_pct,WMAPE_pct,Pred10_pct
5,LightGBM,0.948309,1.857368,54.858680,73.271423,9.527485
7,RandomForest,0.972229,1.844534,55.144062,75.119599,9.999142
6,XGBoost,0.941947,1.861424,55.342941,72.779831,9.235915
3,Croston_SBA,1.020811,2.088825,57.634707,78.873321,7.417889
2,Croston,1.033128,2.082374,58.107241,79.825023,8.850013
1,ARIMA,0.997096,2.054741,58.447490,77.041001,8.532716
0,MA,1.080141,2.554272,64.361831,83.457451,5.548409
4,SeasonalNaive,1.144803,2.341553,81.867816,88.453574,16.079238



Best classical MAPE: 57.63%
Best ML MAPE:        54.86%
MAPE improvement:    +4.8%  (target: 10-25%)
